# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library. It demonstrates step-by-step how to interact with a FAIR Croissant dataset, extract records, and perform basic exploratory data analysis (EDA).

### Dataset Source

The dataset source is provided via a [Croissant schema URL](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json).

In [ ]:
# Ensure `mlcroissant` is installed in the environment
!pip install mlcroissant --quiet

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the Dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset Name: {metadata.name}\n")
print(f"Description: {metadata.description}\n")

## 2. Data Overview

We explore the available record sets (`RecordSet`), their fields, and associated `@id`s using the dataset metadata. This will help us understand which structured tables are included and what columns are available for analysis.

**All entities (record sets, fields, columns) are referenced by their `@id`.**

In [ ]:
from pprint import pprint

# List all record sets by @id
print("Available Record Sets (by @id):\n")
record_sets = list(dataset.record_sets)
record_set_ids = []
for rs in record_sets:
    print(f"- @id: {rs['@id']}, name: {rs['name']}")
    record_set_ids.append(rs['@id'])

print("\nFields per RecordSet (@id and name):\n")
for rs in record_sets:
    print(f"RecordSet @id: {rs['@id']}")
    for field in rs['fields']:
        if isinstance(field, dict):
            print(f"  - Field @id: {field['@id']}, name: {field.get('name','')}, dataType: {field.get('dataType','')} ")
        else:
            print(f"  - Field: {field}")

### Quick sample of first 2 records from one available RecordSet

> **Replace the variable below with the desired RecordSet `@id` from the overview to preview its records.**

In [ ]:
# Choose a record set by @id found above. Adjust as needed.
if len(record_set_ids) > 0:
    record_set_id = record_set_ids[0]
    print(f"Sampling first 2 records from RecordSet: {record_set_id}\n")
    for i, record in enumerate(dataset.records(record_set=record_set_id)):
        pprint(record)
        if i >= 1:
            break
else:
    print("No record sets present in the metadata.")

## 3. Data Extraction

Now we'll extract records from **each** record set into pandas DataFrames for further analysis. Every `RecordSet` is referenced by its `@id` as in the overview. All columns (fields) use their `@id` as well.

In [ ]:
# Collect data from all record sets
dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"RecordSet @{record_set_id}: DataFrame shape = {df.shape}")

# Preview columns of the first RecordSet
if len(record_set_ids) > 0:
    main_rs = record_set_ids[0]
    print(f"\nColumns in RecordSet @{main_rs}:\n{dataframes[main_rs].columns.tolist()}")
    display(dataframes[main_rs].head())

## 4. Exploratory Data Analysis (EDA)

We'll walk through filtering, normalizing and grouping data from a chosen numeric field in a record set. Update the variables to use the appropriate field `@id` values revealed in the previous step.

In [ ]:
# Choose your main RecordSet and Numeric Field @ids
main_record_set_id = record_set_ids[0] if len(record_set_ids) > 0 else None
df = dataframes[main_record_set_id]

# Inspect available columns/fields
print(f"Fields in RecordSet @{main_record_set_id}:")
for i, col in enumerate(df.columns):
    print(f"  {i+1}. {col}")

# Example: Choose a numeric field by its @id. Replace with a real one below as needed.
numeric_field_id = None
possible_numeric = []
for col in df.columns:
    if pd.api.types.is_numeric_dtype(df[col]):
        possible_numeric.append(col)
if possible_numeric:
    numeric_field_id = possible_numeric[0]
    print(f"\nUsing numeric field: {numeric_field_id}\n")
else:
    print("No numeric field detected.")

if numeric_field_id:
    # Example threshold (adjust for your context)
    threshold = df[numeric_field_id].mean()
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records where '@id'={numeric_field_id} > {threshold:.2f} (n={len(filtered_df)})")
    
    # Normalization
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()

    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Example: Try grouping by a likely categorical field (replace as appropriate)
    possible_categorical = [col for col in df.columns if df[col].dtype == 'O' and col != numeric_field_id]
    if possible_categorical:
        group_field = possible_categorical[0]
        print(f"\nGrouping by field: {group_field}\n")
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
        print(grouped_df.head())
    else:
        print("No categorical field available for grouping.")

## 5. Visualization

Let's visualize the distribution of the selected numeric field, as well as how it varies by a grouping variable (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id], kde=True, bins=15)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # Boxplot grouped by the group_field
    if 'group_field' in locals() and group_field in df.columns:
        plt.figure(figsize=(10,4))
        sns.boxplot(data=df, x=group_field, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion

In this notebook, you learned how to load a Croissant FAIR Dataset using the `mlcroissant` Python library, inspect its record sets and fields by `@id`, extract the data into pandas DataFrames, and apply basic EDA and visualization.

**Remember**: always reference fields and tables (`RecordSet`, `Field`, etc.) by their `@id`. Modify the field and record set variables above to suit your exploration of the FAIR² dataset.

Further investigations could include using domain knowledge to filter on specific clinicopathological variables, explore MSI-H prevalence, or prepare data for ML modeling.